In [3]:
import pandas as pd
import numpy as np

# 1. 读取数据（注意文件名是 _100）
file_name = 'pilot_data_raw_100.xlsx'
df = pd.read_excel(file_name)
print(f"✅ 读取成功！原始数据形状: {df.shape}")
print(f"包含列名: {df.columns.tolist()}\n")

# 2. 数据去重 (根据 Job_Url 去除重复的招聘信息)
initial_count = len(df)
df = df.drop_duplicates(subset=['Job_Url'], keep='first')
print(f"🧹 去重完成: 删除了 {initial_count - len(df)} 条重复记录，剩余 {len(df)} 条。\n")

# 3. 缺失值处理
# 查看缺失情况
# print("缺失值情况：\n", df.isnull().sum())

# 填充文本类缺失值
df['Education'] = df['Education'].fillna('Not Specified')
df['Industry'] = df['Industry'].fillna('Unknown')
df['Experience_Min'] = df['Experience_Min'].fillna('Not Specified')
df['Experience_Max'] = df['Experience_Max'].fillna('Not Specified')

# 薪资列保持 NaN (不要用 0 填充，后续分析会算错平均值)
# 如果薪资有字符串格式问题，强制转换为数字，无法转换的变为 NaN
df['Salary_Min'] = pd.to_numeric(df['Salary_Min'], errors='coerce')
df['Salary_Max'] = pd.to_numeric(df['Salary_Max'], errors='coerce')

# 4. 技能列标准化 (确保 Excel, SQL, Python 等列都是 0 或 1 的整数)
skill_cols = ['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'AI_tools']
for col in skill_cols:
    # 如果列里有空值，填充为 0
    df[col] = df[col].fillna(0)
    # 强制转为整数
    df[col] = df[col].astype(int)
print(f"🔧 技能列已标准化为 0/1: {skill_cols}\n")

# 5. 日期格式化
df['Date_Posted'] = pd.to_datetime(df['Date_Posted'], errors='coerce')
df['Date_Collected'] = pd.to_datetime(df['Date_Collected'], errors='coerce')

# 6. 保存清洗后的数据到当前目录 (或者你之前想建的 data/cleaned 文件夹)
# 为了防止因为文件夹不存在报错，这里直接保存为 cleaned 文件
cleaned_file_name = 'pilot_data_cleaned.csv'
df.to_csv(cleaned_file_name, index=False)
print(f"💾 清洗后的数据已保存为: {cleaned_file_name}")
print(f"📊 最终清洗数据形状: {df.shape}")

# 7. 快速检查前 5 行
df.head()

✅ 读取成功！原始数据形状: (100, 25)
包含列名: ['Job_id', 'Country', 'City', 'Job_Title', 'Company_Name', 'Industry', 'Source', 'Job_Url', 'Date_Posted', 'Date_Collected', 'Salary_Min', 'Salary_Max', 'Salary_Currency', 'Experience_Min', 'Experience_Max', 'Education', 'Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'AI_tools', 'JD_text', 'Collector']

🧹 去重完成: 删除了 0 条重复记录，剩余 100 条。

🔧 技能列已标准化为 0/1: ['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'AI_tools']

💾 清洗后的数据已保存为: pilot_data_cleaned.csv
📊 最终清洗数据形状: (100, 25)


,Job_id,Country,City,Job_Title,Company_Name,Industry,Source,Job_Url,Date_Posted,Date_Collected,...,Education,Excel,SQL,Python,R,Tableau,Power_BI,AI_tools,JD_text,Collector
0,BA0001,United States,"Atlanta, GA",Staff Business Analyst,TriNet,Technology,LinkedIn,https://www.linkedin.com/jobs/view/staff-busin...,2026-07-16,2026-09-15,...,Bachelor's,0,0,0,0,0,0,0,TriNet is a leading provider of comprehensive ...,A
1,BA0002,United States,"Reston, VA",Business Analyst (ServiceNow) - (High Level Cl...,ICF,Technology,LinkedIn,https://www.linkedin.com/jobs/view/business-an...,2026-09-14,2026-09-15,...,Bachelor's,0,0,0,0,0,0,1,Description ICF is seeking a motivated Busines...,A
2,BA0003,United States,"Miami, FL","Business Analyst, Fuse Billing Operations",Amazon,Technology,LinkedIn,https://www.linkedin.com/jobs/view/business-an...,2026-09-09,2026-09-15,...,Bachelor's,1,1,0,0,1,0,0,Description The Amazon Mobile Business Develop...,A
3,BA0004,United States,"Boston, MA",Systems Business Analyst Sr.,Brown Brothers Harriman,Technology,LinkedIn,https://www.linkedin.com/jobs/view/systems-bus...,2026-09-17,2026-09-15,...,Not Specified,1,1,0,0,0,0,0,"At BBH, Partnership is more than a form of own...",A
4,BA0005,United States,"Las Vegas, NV",Sr Business Analyst,Las Vegas Sands Corp.,Technology,LinkedIn,https://www.linkedin.com/jobs/view/sr-business...,2026-09-16,2026-09-15,...,Bachelor's,1,1,0,0,0,0,0,Job Description: Position Overview The primary...,A


In [4]:
# 查看薪资基本统计量
print(df[['Salary_Min', 'Salary_Max']].describe())

# 查看技能标签的平均值（即“要求该技能的比例”）
print("\n各技能要求比例：")
print(df[['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'AI_tools']].mean())

          Salary_Min     Salary_Max
count      33.000000      34.000000
mean    81005.909091  121295.911765
std     22265.918639   35307.533377
min     42000.000000   50000.000000
25%     65100.000000  103688.000000
50%     80000.000000  120000.000000
75%    100000.000000  141887.250000
max    126000.000000  218520.000000

各技能要求比例：
Excel       0.29
SQL         0.21
Python      0.02
R           0.00
Tableau     0.04
Power_BI    0.09
AI_tools    0.12
dtype: float64
